In [ ]:
import io
import os
import gzip
import pandas as pd
import numpy as np
import scanpy as sc
import matplotlib.pyplot as plt
from scipy import sparse
from nichenetpy.prediction import LigandActivityPredictor
from nichenetpy.network import LigandReceptorNetwork, WeightedNetwork
from nichenetpy.io import read_ligand_target_matrix
from nichenetpy.extraction import get_expressed_genes
from nichenetpy.wrappers import (
    combine_weighted_ligand_target_links,
    create_regulatory_potential_heatmap
)


In [ ]:
def save_fig(
    fig_name,
    save_dir="/data/zdh/ZDH_Spatial/Kidney_Spatial_Project/ceshi/Figure/20260922",
    fmt="pdf",
    dpi=300,
):
    save_fig.counter = getattr(save_fig, "counter", 1)
    filename = f"{save_fig.counter:02d}_{fig_name}.{fmt}"
    plt.savefig(
        os.path.join(save_dir, filename),
        format=fmt,
        dpi=dpi,
        bbox_inches="tight",
    )
    save_fig.counter += 1

In [ ]:

SAVE_DIR = "/data/zdh/ZDH_Spatial/Kidney_Spatial_Project/ceshi/Figure/20260922"
MODEL_PATH = "/data/zdh/ZDH_Spatial/Python包"
os.makedirs(SAVE_DIR, exist_ok=True)


In [ ]:
adata_xeno = sc.read(os.path.join(SAVE_DIR, "adata_xeno_pf4_major.h5ad"))
adata_allo = sc.read(os.path.join(SAVE_DIR, "adata_allo_pf4_major.h5ad"))


In [ ]:

adata = sc.concat(
    [adata_xeno, adata_allo],
    join="inner",
    merge="same",
    label="aggregate",
    keys=["Xeno D33", "Allo ABMR"],
    index_unique="-"
)
adata.var_names_make_unique()
adata.obs_names_make_unique()
adata.obs["celltype"] = adata.obs["celltype_coarse_2"].astype(str)


In [ ]:

predictor = LigandActivityPredictor(
    *read_ligand_target_matrix(os.path.join(MODEL_PATH, "ligand_target_matrix_human.bin"))
)
lr_network = LigandReceptorNetwork(filename=os.path.join(MODEL_PATH, "lr_network_human.bin"))
lr_sig = WeightedNetwork(filename=os.path.join(MODEL_PATH, "lr_sig_human.bin"))


In [ ]:

adata_xeno_niche = adata[adata.obs["aggregate"] == "Xeno D33"].copy()


In [ ]:

expressed_genes_sender = set(
    get_expressed_genes("PF4+PPBP+ Mono", adata_xeno_niche, pct=0.05, celltype_col="celltype", layer="lognorm")
)


In [ ]:
expressed_genes_receiver = set(
    get_expressed_genes("DC", adata_xeno_niche, pct=0.05, celltype_col="celltype", layer="lognorm")
)


In [ ]:

all_ligands = lr_network.get_ligands()
all_receptors = lr_network.get_receptors()


In [ ]:
expressed_ligands = all_ligands & expressed_genes_sender
expressed_receptors = all_receptors & expressed_genes_receiver


In [ ]:

potential_ligands = {
    ligand for ligand, receptor in lr_network
    if ligand in expressed_ligands and receptor in expressed_receptors
}


In [ ]:

top10_lr = [
    ("CD55_ADGRE5", "CD55", "ADGRE5"),
    ("PECAM1_CD38", "PECAM1", "CD38"),
    ("LPAR1_ADGRE5", "LPAR1", "ADGRE5"),
    ("SELPLG_SELL", "SELPLG", "SELL"),
    ("THBS1_CD36", "THBS1", "CD36"),
    ("TGFB1_TGFbeta_receptor2", "TGFB1", None),
    ("SEMA4D_PLXNB2", "SEMA4D", "PLXNB2"),
    ("TNFSF10_TNFRSF10D", "TNFSF10", "TNFRSF10D"),
    ("TNFSF10_TNFRSF10A", "TNFSF10", "TNFRSF10A"),
    ("RNASET2_TLR8", "RNASET2", "TLR8")
]


In [ ]:

lr_prior = {ligand: set(receptors) for ligand, receptors in lr_network.item_iter()}
final_candidate_ligands = {
    lig for _, lig, rec in top10_lr
    if lig in potential_ligands and (rec in expressed_receptors or rec is None)
}


In [ ]:

soft_path = os.path.join(MODEL_PATH, "GPL30305_family.soft.gz")
with gzip.open(soft_path, "rt", encoding="utf-8", errors="replace") as f:
    txt = f.read()

start = txt.index("!platform_table_begin") + len("!platform_table_begin")
end = txt.index("!platform_table_end")
bhot_platform = pd.read_csv(io.StringIO(txt[start:end].strip()), sep="\t")


In [ ]:

technical_controls = (
    bhot_platform["ID"].str.startswith("NEG_", na=False) |
    bhot_platform["ID"].str.startswith("POS_", na=False)
)
bhot_genes = set(bhot_platform.loc[~technical_controls, "ID"].dropna().astype(str))


In [ ]:

nichenet_genes = set(predictor.get_genes())
bhot_xeno_dc_nichenet = sorted(bhot_genes & set(adata.var_names) & expressed_genes_receiver & nichenet_genes)


In [ ]:

xeno_dc = adata[(adata.obs["aggregate"] == "Xeno D33") & (adata.obs["celltype"] == "DC")]
allo_dc = adata[(adata.obs["aggregate"] == "Allo ABMR") & (adata.obs["celltype"] == "DC")]


In [ ]:

X_xeno = xeno_dc[:, bhot_xeno_dc_nichenet].layers["lognorm"]
X_allo = allo_dc[:, bhot_xeno_dc_nichenet].layers["lognorm"]


In [ ]:

mean_xeno = np.asarray(X_xeno.mean(axis=0)).ravel() if sparse.issparse(X_xeno) else np.asarray(X_xeno).mean(axis=0)
mean_allo = np.asarray(X_allo.mean(axis=0)).ravel() if sparse.issparse(X_allo) else np.asarray(X_allo).mean(axis=0)


In [ ]:

bhot_dc_direction = pd.DataFrame({
    "gene": bhot_xeno_dc_nichenet,
    "mean_xeno": mean_xeno,
    "mean_allo": mean_allo,
    "delta_mean_logexpr": mean_xeno - mean_allo
}).sort_values("delta_mean_logexpr", ascending=False)


In [ ]:

geneset_oi = set(bhot_dc_direction.query("delta_mean_logexpr > 0").head(100)["gene"])
background_expressed_genes = expressed_genes_receiver & nichenet_genes


In [ ]:

ligand_activities = predictor.predict_ligand_activities(
    geneset_oi,
    background_expressed_genes,
    final_candidate_ligands
)


In [ ]:

ligand_activity_df = (
    pd.DataFrame.from_dict(ligand_activities, orient="index")
    .reset_index()
    .rename(columns={"index": "ligand"})
    .sort_values("aupr_corrected", ascending=False)
    .reset_index(drop=True)
)
ligand_activity_df["rank"] = np.arange(1, len(ligand_activity_df) + 1)
display(ligand_activity_df)


In [ ]:

best_ligands = ligand_activity_df.sort_values("aupr_corrected", ascending=False)["ligand"].tolist()
target_link_dicts = [
    predictor.get_weighted_ligand_target_links(lig, geneset_oi, n=200)
    for lig in best_ligands
]
active_ligand_target_links = combine_weighted_ligand_target_links(target_link_dicts)


In [ ]:

ligand_target_df = (
    pd.DataFrame(active_ligand_target_links, columns=["ligand", "target", "weight"])
    .sort_values(["ligand", "weight"], ascending=[True, False])
    .reset_index(drop=True)
)
display(ligand_target_df.head(20))


Fgiure 6A

In [ ]:

fig, ax = create_regulatory_potential_heatmap(
    predictor,
    active_ligand_target_links,
    figsize=(12, 5)
)
save_fig("")
plt.show()

Fgiure 6B

In [ ]:
import seaborn as sns
from matplotlib.gridspec import GridSpec

ligand_order = (
    ligand_activity_df.sort_values("pearson", ascending=False)["ligand"].tolist()
)

activity_mat = ligand_activity_df.set_index("ligand").loc[
    ligand_order, ["pearson"]
]
activity_mat.columns = ["Ligand activity"]

target_mat = (
    ligand_target_df.pivot_table(
        index="ligand",
        columns="target",
        values="weight",
        aggfunc="max",
        fill_value=0,
    )
).reindex(ligand_order)

target_order = target_mat.max(axis=0).sort_values(ascending=False).index
target_mat = target_mat[target_order]

fig = plt.figure(figsize=(14, 5.8), dpi=300)
gs = GridSpec(1, 2, width_ratios=[1.15, 7.0], wspace=0.08)

ax1 = fig.add_subplot(gs[0, 0])
ax2 = fig.add_subplot(gs[0, 1])

sns.heatmap(
    activity_mat,
    cmap="Oranges",
    annot=True,
    fmt=".3f",
    linewidths=0.8,
    linecolor="white",
    cbar_kws={"label": "Pearson correlation", "shrink": 0.55, "pad": 0.08},
    ax=ax1,
)
ax1.set_xlabel("")
ax1.set_ylabel("")
ax1.set_xticklabels(["Ligand\nactivity"], rotation=0, fontsize=10)
ax1.tick_params(axis="y", labelrotation=0, labelsize=10)

for tick in ax1.get_yticklabels():
  if tick.get_text() == "THBS1":
    tick.set_color("#C23B32")
    tick.set_fontweight("bold")

sns.heatmap(
    target_mat,
    cmap="Purples",
    linewidths=0.6,
    linecolor="white",
    cbar_kws={"label": "Regulatory potential", "shrink": 0.55, "pad": 0.02},
    yticklabels=False,
    ax=ax2,
)
ax2.set_xlabel("Predicted target genes", fontsize=11)
ax2.set_ylabel("")
ax2.tick_params(axis="x", rotation=90, labelsize=8)

fig.suptitle(
    "NicheNet prioritization of PF4+PPBP+ Mono ligands and predicted DC targets",
    fontsize=14,
    fontweight="bold",
    y=1.02,
)
plt.tight_layout()
save_fig("")
plt.show()